# 🦙 Forecasting com Lag-Llama

**Modelo**: Lag-Llama (Foundation Model — ServiceNow Research / Morgan Stanley)
**Séries**: 29 séries econômicas brasileiras
**Horizonte**: 3 meses

---

## Características do Lag-Llama:
- Foundation Model com arquitetura **decoder-only** (similar ao LLaMA, porém desenvolvido pela ServiceNow Research / Morgan Stanley)
- Treinado em dados de séries temporais de diversas fontes
- Suporta **zero-shot** e **fine-tuning**
- Utiliza **lags** como features (daí o nome)
- Previsão probabilística (amostras)

**Paper**: "Lag-Llama: Towards Foundation Models for Probabilistic Time Series Forecasting" — Rasul et al. (2023), ServiceNow Research / Morgan Stanley

---

**Ficha Técnica do Modelo**

| Campo | Valor |
|-------|-------|
| **Modelo** | Lag-Llama — Foundation Model decoder-only para séries temporais (zero-shot) |
| **Biblioteca** | `lag-llama` (local) + `gluonts` — `LagLlamaEstimator` |
| **Versão do modelo** | `lag-llama.ckpt` (checkpoint local, arquitetura decoder-only com RoPE scaling) |
| **Hiperparâmetros configurados** | `CONTEXT_LENGTH=32`, `num_parallel_samples=100`, `batch_size=1`, `rope_scaling={'type':'linear'}`, `SEED=42` |
| **Busca de hiperparâmetros** | Não (zero-shot — sem fine-tuning) |
| **Critério de seleção** | N/A |
| **Séries utilizadas** | 29 séries com treino ≥ 32 observações (`len(train_series) < CONTEXT_LENGTH`) |
| **Horizonte** | 3 meses (`HORIZON = 3`) |
| **Protocolo de avaliação** | Walk-forward expansível, 24 meses de teste (`TEST_SIZE = 24`), janelas de 3 meses; previsão probabilística com mediana de 100 amostras |
| **Reprodutibilidade** | `SEED = 42` (`random.seed` + `np.random.seed` + `torch.manual_seed` + `cuda.manual_seed_all`) |
| **Referência** | Rasul, K. et al. (2023). Lag-Llama: Towards Foundation Models for Probabilistic Time Series Forecasting. *arXiv:2310.08278*. **ServiceNow Research / Morgan Stanley**. |

---

In [ ]:
# ── Semente global para reprodutibilidade ──
import random, numpy as np, torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"🔒 Seed fixada: {SEED} (random + numpy + torch)")

## 1. Importação de Bibliotecas

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
import warnings
from datetime import datetime

warnings.filterwarnings('ignore')

print("="*70)
print("🦙 FORECASTING COM LAG-LLAMA")
print("="*70)
print(f"📅 Data de execução: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🖥️ Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

🦙 FORECASTING COM LAG-LLAMA
📅 Data de execução: 2026-04-05 13:44:32
🖥️ Device: cuda


## 2. Carregamento dos Dados

In [2]:
# Carregar base de dados
df = pd.read_csv('base_economica_brasil.csv', index_col=0, parse_dates=True)

print(f"\n📊 BASE DE DADOS:")
print(f"   • Shape: {df.shape}")
print(f"   • Período: {df.index.min().strftime('%Y-%m')} a {df.index.max().strftime('%Y-%m')}")
print(f"\n📈 Séries disponíveis ({len(df.columns)}):")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2}. {col}")


📊 BASE DE DADOS:
   • Shape: (108, 35)
   • Período: 2017-01 a 2025-12

📈 Séries disponíveis (35):
    1. IBC_Br
    2. Selic
    3. Cambio_USDBRL
    4. Desemprego
    5. Brent_USD
    6. Soja_USD
    7. Minerio_USD
    8. Ibovespa
    9. ICC_FGV
   10. Credito_Total
   11. Inadimplencia
   12. Massa_Salarial
   13. CPI_USA
   14. Prod_Ind_USA
   15. Cafe_USD
   16. Ouro_USD
   17. GasNatural_USD
   18. Cobre_USD
   19. ETF_Emergentes
   20. IGP_DI
   21. INCC
   22. ICE_Empresarial
   23. Housing_Starts_EUA
   24. Dollar_Index_Fed
   25. PMS_Volume
   26. PMC_Ampliado
   27. IGPM
   28. INPC
   29. M2
   30. Divida_PIB
   31. Vendas_Varejo
   32. Balanca_Comercial
   33. NUCI_FGV
   34. EAI_Emprego_Ind
   35. SP500


## 3. Configuração do Experimento

In [3]:
from gluonts.dataset.common import ListDataset
from lag_llama.gluon.estimator import LagLlamaEstimator

# Configurações
HORIZON = 3              # Prever 3 meses
CONTEXT_LENGTH = 32      # Contexto para o modelo
CKPT_PATH = 'lag-llama.ckpt'

print("📐 Configuração:")
print(f"   • Horizonte: {HORIZON} meses")
print(f"   • Contexto: {CONTEXT_LENGTH} observações")
print(f"   • Checkpoint: {CKPT_PATH}")

📐 Configuração:
   • Horizonte: 3 meses
   • Contexto: 32 observações
   • Checkpoint: lag-llama.ckpt


In [4]:
# Carregar modelo (método igual ao Tutorial IBM)
print("\n⏳ Carregando modelo Lag-Llama...")

import functools

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Carregar checkpoint e extrair hyperparameters (método IBM)
ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
estimator_args = ckpt["hyper_parameters"]["model_kwargs"]

print("📋 Parâmetros do checkpoint:")
for k, v in list(estimator_args.items())[:8]:
    print(f"   {k}: {v}")

# Patch para weights_only (PyTorch 2.6+)
_original_torch_load = torch.load
@functools.wraps(_original_torch_load)
def _patched_torch_load(*args, **kwargs):
    kwargs.setdefault('weights_only', False)
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

# Criar estimator com parâmetros do checkpoint
estimator = LagLlamaEstimator(
    ckpt_path=CKPT_PATH,
    prediction_length=HORIZON,
    context_length=CONTEXT_LENGTH,
    device=device,
    
    # Parâmetros do checkpoint
    input_size=estimator_args["input_size"],
    n_layer=estimator_args["n_layer"],
    n_embd_per_head=estimator_args["n_embd_per_head"],
    n_head=estimator_args["n_head"],
    scaling=estimator_args["scaling"],
    time_feat=estimator_args["time_feat"],
    
    # RoPE scaling para contextos maiores
    rope_scaling={
        "type": "linear",
        "factor": max(1.0, (CONTEXT_LENGTH + HORIZON) / estimator_args["context_length"]),
    },
    
    batch_size=1,
    num_parallel_samples=100
)

predictor = estimator.create_predictor(
    estimator.create_transformation(),
    estimator.create_lightning_module()
)

# Restaurar torch.load original
torch.load = _original_torch_load

print("✅ Modelo carregado!")


⏳ Carregando modelo Lag-Llama...
📋 Parâmetros do checkpoint:
   input_size: 1
   context_length: 32
   max_context_length: 2048
   lags_seq: [0, 7, 8, 10, 11, 12, 13, 14, 19, 20, 21, 22, 23, 24, 26, 27, 28, 29, 30, 34, 35, 36, 46, 47, 48, 50, 51, 52, 55, 57, 58, 59, 60, 61, 70, 71, 72, 83, 94, 95, 96, 102, 103, 104, 117, 118, 119, 120, 121, 142, 143, 144, 154, 155, 156, 166, 167, 168, 177, 178, 179, 180, 181, 334, 335, 336, 362, 363, 364, 502, 503, 504, 670, 671, 672, 718, 719, 720, 726, 727, 728, 1090, 1091, 1092]
   n_layer: 8
   n_embd_per_head: 16
   n_head: 9
   scaling: robust
✅ Modelo carregado!


In [5]:
def calculate_metrics(y_true, y_pred):
    """
    Calcula métricas de avaliação.
    """
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()
    
    mse = np.mean((y_true - y_pred)**2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    
    return {'MSE': mse, 'RMSE': rmse, 'MAE': mae, 'MAPE': mape}

print("✅ Funções auxiliares definidas!")

✅ Funções auxiliares definidas!


In [ ]:
# ============================================
# DIVISÃO TREINO/TESTE
# ============================================
# Últimos 24 períodos para teste, restante para treino (igual aos outros modelos)

TEST_SIZE = 24  # últimos 24 meses para teste

train_df = df.iloc[:-TEST_SIZE]
test_df = df.iloc[-TEST_SIZE:]

print("=" * 60)
print("📊 DIVISÃO TREINO/TESTE")
print("=" * 60)
print(f"\n🔹 TREINO:")
print(f"   Período: {train_df.index.min().strftime('%Y-%m')} a {train_df.index.max().strftime('%Y-%m')}")
print(f"   Observações: {len(train_df)}")

print(f"\n🔹 TESTE:")
print(f"   Período: {test_df.index.min().strftime('%Y-%m')} a {test_df.index.max().strftime('%Y-%m')}")
print(f"   Observações: {len(test_df)}")

print(f"\n📈 Proporção: {len(train_df)/(len(df))*100:.1f}% treino / {len(test_df)/(len(df))*100:.1f}% teste")

📊 DIVISÃO TREINO/TESTE

🔹 TREINO:
   Período: 2017-01 a 2024-12
   Observações: 96

🔹 TESTE:
   Período: 2025-01 a 2025-12
   Observações: 12

📈 Proporção: 88.9% treino / 11.1% teste


## 4. Treinamento e Previsão (Walk-Forward)

In [7]:
print("\n" + "="*80)
print(f"🦙 EXECUTANDO LAG-LLAMA (WALK-FORWARD MULTI-STEP, horizon={HORIZON})")
print(f"   Treino: {train_df.index.min().strftime('%Y-%m')} a {train_df.index.max().strftime('%Y-%m')} ({len(train_df)} obs)")
print(f"   Teste:  {test_df.index.min().strftime('%Y-%m')} a {test_df.index.max().strftime('%Y-%m')} ({len(test_df)} obs)")
print("="*80)

ALL_SERIES = list(df.columns)
# Excluir PIM e IPCA_mensal da análise
ALL_SERIES = [s for s in ALL_SERIES if s not in ['PIM', 'IPCA_mensal']]
all_results = {}

for idx, series_name in enumerate(ALL_SERIES, 1):
    print(f"\n[{idx}/{len(ALL_SERIES)}] 📈 {series_name}")
    print("-"*50)
    
    try:
        # Usar divisão treino/teste
        train_series = train_df[series_name].dropna()
        test_series = test_df[series_name].dropna()
        
        if len(train_series) < CONTEXT_LENGTH:
            print(f"   ⚠️ Poucos dados de treino: {len(train_series)} obs")
            continue
        
        if len(test_series) == 0:
            print(f"   ⚠️ Sem dados de teste")
            continue
        
        forecasts = []
        actuals = []
        dates = []
        
        # Walk-forward multi-step: janelas de HORIZON passos (4 × 3 = 12 pontos)
        for i in range(0, len(test_series), HORIZON):
            n_steps = min(HORIZON, len(test_series) - i)
            
            # Dados históricos: treino + pontos de teste anteriores
            if i == 0:
                history = train_series.copy()
            else:
                history = pd.concat([train_series, test_series.iloc[:i]])
            
            # Criar dataset GluonTS
            dataset = ListDataset(
                [{'start': history.index[0], 'target': history.values}],
                freq='M'
            )
            
            # Previsão de HORIZON passos à frente
            forecast_it = predictor.predict(dataset)
            forecast = list(forecast_it)[0]
            preds = np.median(forecast.samples, axis=0)[:n_steps]  # Pegar os n_steps passos
            
            for j in range(n_steps):
                forecasts.append(preds[j])
                actuals.append(test_series.iloc[i + j])
                dates.append(test_series.index[i + j])
        
        # Calcular métricas
        forecasts_arr = np.array(forecasts)
        actuals_arr = np.array(actuals)
        
        mae = np.mean(np.abs(actuals_arr - forecasts_arr))
        rmse = np.sqrt(np.mean((actuals_arr - forecasts_arr) ** 2))
        mape = np.mean(np.abs((actuals_arr - forecasts_arr) / (actuals_arr + 1e-8))) * 100
        
        print(f"   ✅ MAE={mae:.2f} | RMSE={rmse:.2f} | MAPE={mape:.2f}%")
        
        # Guardar resultados
        all_results[series_name] = {
            'mae': mae,
            'rmse': rmse,
            'mape': mape,
            'n_points': len(forecasts),
            'forecasts': forecasts,
            'actuals': actuals,
            'dates': dates,
            'train_series': train_series,
            'test_series': test_series
        }
        
    except Exception as e:
        print(f"   ❌ Erro: {str(e)[:60]}")

print("\n" + "="*80)
print(f"✅ CONCLUÍDO: {len(all_results)}/{len(ALL_SERIES)} séries processadas")
print("="*80)


🦙 EXECUTANDO LAG-LLAMA (WALK-FORWARD MULTI-STEP, horizon=3)
   Treino: 2017-01 a 2024-12 (96 obs)
   Teste:  2025-01 a 2025-12 (12 obs)

[1/35] 📈 IBC_Br
--------------------------------------------------
   ✅ MAE=9.18 | RMSE=10.25 | MAPE=8.36%

[2/35] 📈 Selic
--------------------------------------------------
   ✅ MAE=0.94 | RMSE=1.08 | MAPE=6.52%

[3/35] 📈 Cambio_USDBRL
--------------------------------------------------
   ✅ MAE=0.24 | RMSE=0.27 | MAPE=4.28%

[4/35] 📈 Desemprego
--------------------------------------------------
   ✅ MAE=1.18 | RMSE=1.30 | MAPE=20.25%

[5/35] 📈 Brent_USD
--------------------------------------------------
   ✅ MAE=6.46 | RMSE=7.56 | MAPE=9.80%

[6/35] 📈 Soja_USD
--------------------------------------------------
   ✅ MAE=26.12 | RMSE=33.16 | MAPE=6.87%

[7/35] 📈 Minerio_USD
--------------------------------------------------
   ✅ MAE=5.90 | RMSE=7.18 | MAPE=5.71%

[8/35] 📈 Ibovespa
--------------------------------------------------
   ✅ MAE=13058.81 | 

## 5. Resultados e Métricas

In [8]:
# Criar DataFrame com resultados
results_df = pd.DataFrame([
    {
        'Série': name,
        'MAE': data['mae'],
        'RMSE': data['rmse'],
        'MAPE (%)': data['mape'],
        'N Pontos': data['n_points']
    }
    for name, data in all_results.items()
]).sort_values('MAPE (%)')

# Adicionar classificação baseada no MAPE
def classificar(mape):
    if mape < 10:
        return '⭐ Excelente'
    elif mape < 20:
        return '✅ Muito Bom'
    elif mape < 30:
        return '👍 Bom'
    elif mape < 50:
        return '⚠️ Regular'
    else:
        return '❌ Difícil'

results_df['Classificação'] = results_df['MAPE (%)'].apply(classificar)
results_df = results_df.set_index('Série')

# Exibir tabela
print("="*80)
print("📊 RANKING - LAG-LLAMA (MAE, RMSE, MAPE)")
print("="*80)
print(f"\nModelo: Lag-Llama | Horizonte: {HORIZON} meses\n")
print(results_df.round(2).to_string())

# Estatísticas gerais
print("\n" + "-"*80)
print("📈 ESTATÍSTICAS GERAIS:")
print(f"   Total de séries analisadas: {len(results_df)}")
print(f"\n   📉 MAE (Mean Absolute Error):")
print(f"      Média geral: {results_df['MAE'].mean():.2f}")
print(f"      Melhor: {results_df['MAE'].idxmin()} ({results_df['MAE'].min():.2f})")
print(f"      Pior: {results_df['MAE'].idxmax()} ({results_df['MAE'].max():.2f})")
print(f"\n   📉 RMSE (Root Mean Squared Error):")
print(f"      Média geral: {results_df['RMSE'].mean():.2f}")
print(f"      Melhor: {results_df['RMSE'].idxmin()} ({results_df['RMSE'].min():.2f})")
print(f"      Pior: {results_df['RMSE'].idxmax()} ({results_df['RMSE'].max():.2f})")
print(f"\n   📉 MAPE (Mean Absolute Percentage Error):")
print(f"      Média geral: {results_df['MAPE (%)'].mean():.2f}%")
print(f"      Melhor: {results_df['MAPE (%)'].idxmin()} ({results_df['MAPE (%)'].min():.2f}%)")
print(f"      Pior: {results_df['MAPE (%)'].idxmax()} ({results_df['MAPE (%)'].max():.2f}%)")
print(f"      Séries com MAPE < 10%: {(results_df['MAPE (%)'] < 10).sum()}")
print(f"      Séries com MAPE < 20%: {(results_df['MAPE (%)'] < 20).sum()}")

📊 RANKING - LAG-LLAMA (MAE, RMSE, MAPE)

Modelo: Lag-Llama | Horizonte: 3 meses

                             MAE          RMSE      MAPE (%)  N Pontos Classificação
Série                                                                               
Prod_Ind_USA        1.180000e+00  1.420000e+00  1.160000e+00        12   ⭐ Excelente
ICC_FGV             3.700000e+00  4.940000e+00  3.140000e+00        12   ⭐ Excelente
NUCI_FGV            3.220000e+00  3.860000e+00  3.920000e+00        12   ⭐ Excelente
Housing_Starts_EUA  5.363000e+01  6.093000e+01  3.950000e+00        12   ⭐ Excelente
SP500               2.706900e+02  3.514100e+02  4.250000e+00        12   ⭐ Excelente
Cambio_USDBRL       2.400000e-01  2.700000e-01  4.280000e+00        12   ⭐ Excelente
Credito_Total       1.770628e+05  2.019521e+05  4.520000e+00        12   ⭐ Excelente
Dollar_Index_Fed    5.660000e+00  6.320000e+00  4.580000e+00        12   ⭐ Excelente
Minerio_USD         5.900000e+00  7.180000e+00  5.710000e+00        1

## 6. Visualização: Ranking MAPE por Série

In [ ]:
# ── Gráfico: Ranking MAPE por Série ──
sorted_df = results_df.sort_values('MAPE (%)')

fig, ax = plt.subplots(figsize=(12, 8))

cores = ['#2ecc71' if m < 10 else '#3498db' if m < 20 else '#f39c12' if m < 30 else '#e74c3c'
         for m in sorted_df['MAPE (%)']]

bars = ax.barh(range(len(sorted_df)), sorted_df['MAPE (%)'],
               color=cores, edgecolor='white', height=0.7)
ax.set_yticks(range(len(sorted_df)))
ax.set_yticklabels(sorted_df.index)
ax.invert_yaxis()
ax.set_xlabel('MAPE (%)')
ax.set_title(f'Lag-Llama — MAPE por Série\n(Walk-Forward, h={HORIZON}, teste={TEST_SIZE} meses)',
             fontsize=13, fontweight='bold')
ax.axvline(x=sorted_df['MAPE (%)'].mean(), color='red', linestyle='--',
           label=f'Média: {sorted_df["MAPE (%)"].mean():.1f}%')
ax.legend(loc='lower right')

for i, (bar, val) in enumerate(zip(bars, sorted_df['MAPE (%)'])):
    ax.text(val + 0.5, i, f'{val:.1f}%', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('lagllama_mape_por_serie.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: lagllama_mape_por_serie.png")

## 7. Visualização: Real vs. Projetado (Top 6 Séries)

In [ ]:
# ── Gráfico: Real vs. Projetado (Top 6 Séries por MAPE) ──
top6 = sorted_df.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for ax, sn in zip(axes.flatten(), top6):
    data = all_results[sn]

    # Contexto: últimos 24 meses de treino
    train_ctx = data['train_series'].iloc[-24:]
    ax.plot(train_ctx.index, train_ctx.values, 'b-',
            label='Treino', linewidth=1, alpha=0.5)
    ax.axvline(x=train_ctx.index[-1], color='gray', linestyle='--', alpha=0.5)

    # Valores reais (teste)
    ax.plot(data['dates'], data['actuals'], 'b-o',
            label='Real', markersize=4, linewidth=2)

    # Previsões do modelo
    ax.plot(data['dates'], data['forecasts'], 'r--s',
            label='Previsão', markersize=4, linewidth=2)

    ax.set_title(f"{sn}\nMAPE: {data['mape']:.1f}%", fontsize=10)
    ax.grid(True, alpha=0.3)
    ax.tick_params(axis='x', rotation=45, labelsize=8)

axes.flatten()[0].legend(fontsize=8)
fig.suptitle('Lag-Llama — Real vs. Projetado (6 Melhores Séries)',
             fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig('lagllama_previsoes.png', dpi=150, bbox_inches='tight')
plt.show()
print("📊 Gráfico salvo: lagllama_previsoes.png")

## 8. Exportação de Resultados

In [11]:
# Salvar resultados com nomes padronizados para compatibilidade com consolidação
results_export = results_df.reset_index()
results_export.columns = ['Serie', 'MAE', 'RMSE', 'MAPE', 'N_Pontos', 'Classificacao']
results_export.to_csv('resultados_lagllama.csv', index=False)
print("💾 Resultados salvos em 'resultados_lagllama.csv'")
print(f"   Colunas: {list(results_export.columns)}")

# Salvar previsões individuais (Serie, Data, Previsao) para análises complementares
previsoes_rows = []
for serie, data in all_results.items():
    for d, p in zip(data['dates'], data['forecasts']):
        previsoes_rows.append({'Serie': serie, 'Data': str(d)[:10], 'Previsao': p})
df_prev = pd.DataFrame(previsoes_rows)
df_prev.to_csv('previsoes_lagllama.csv', index=False)
print(f"💾 Previsões salvas em 'previsoes_lagllama.csv' ({len(df_prev)} linhas)")

💾 Resultados salvos em 'resultados_lagllama.csv'
   Colunas: ['Serie', 'MAE', 'RMSE', 'MAPE', 'N_Pontos', 'Classificacao']
💾 Previsões salvas em 'previsoes_lagllama.csv' (420 linhas)
